# Topic 17: Decision Trees: Entropy, Information Gain, and Why One Tree Overfits

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 17 lecture note](https://dongchul.kim/csci3320-fall2026/topic/18), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## A tree is just if/else

In [ ]:
def will_play(outlook, humidity, wind):
    if outlook == "Overcast":         # (1) all 4 Overcast days were Yes
        return "Yes"
    if outlook == "Sunny":            # (2) Sunny days: ask about humidity
        if humidity == "High":
            return "No"
        return "Yes"
    if outlook == "Rain":             # (3) Rainy days: ask about wind
        if wind == "Strong":
            return "No"
        return "Yes"

print(will_play("Sunny", "High", "Strong"))      # day 15
print(will_play("Overcast", "High", "Strong"))
print(will_play("Rain", "Normal", "Weak"))

## Entropy and information gain in code

In [ ]:
import math
from collections import Counter
import pandas as pd

rows = [
    ("Sunny", "Hot", "High", "Weak", "No"),        ("Sunny", "Hot", "High", "Strong", "No"),
    ("Overcast", "Hot", "High", "Weak", "Yes"),    ("Rain", "Mild", "High", "Weak", "Yes"),
    ("Rain", "Cool", "Normal", "Weak", "Yes"),     ("Rain", "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"), ("Sunny", "Mild", "High", "Weak", "No"),
    ("Sunny", "Cool", "Normal", "Weak", "Yes"),    ("Rain", "Mild", "Normal", "Weak", "Yes"),
    ("Sunny", "Mild", "Normal", "Strong", "Yes"),  ("Overcast", "Mild", "High", "Strong", "Yes"),
    ("Overcast", "Hot", "Normal", "Weak", "Yes"),  ("Rain", "Mild", "High", "Strong", "No"),
]
df = pd.DataFrame(rows, columns=["Outlook", "Temp", "Humidity", "Wind", "Play"])

def entropy(labels):
    n = len(labels)
    counts = Counter(labels)                       # e.g. {"Yes": 9, "No": 5}
    return -sum(c / n * math.log2(c / n) for c in counts.values())

def info_gain(df, feature, target="Play"):
    before = entropy(df[target])
    after = sum(len(g) / len(df) * entropy(g[target])
                for _, g in df.groupby(feature))   # one group per answer
    return before - after

print(f"entropy of all 14 days: {entropy(df['Play']):.3f}")
for f in ["Outlook", "Temp", "Humidity", "Wind"]:
    print(f"{f:9s} gain {info_gain(df, f):.3f}")

sunny = df[df["Outlook"] == "Sunny"]               # only the 5 Sunny days
print("inside Sunny:")
for f in ["Temp", "Humidity", "Wind"]:
    print(f"  {f:9s} gain {info_gain(sunny, f):.3f}")

## Another way to measure "mixed": Gini impurity

In [ ]:
import math

def entropy_of(counts):
    n = sum(counts)
    return sum(-c / n * math.log2(c / n) for c in counts if c > 0)

def gini_of(counts):
    n = sum(counts)
    return 1 - sum((c / n) ** 2 for c in counts)

for counts in [(4, 0), (3, 3), (9, 5), (2, 3)]:
    print(f"{str(counts):8s} entropy {entropy_of(counts):.3f}   Gini {gini_of(counts):.3f}")

## A tree on Breast cancer

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X, y = data.data, data.target                  # y: 0 = malignant, 1 = benign
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0)

tree = DecisionTreeClassifier(random_state=0)  # no limits
tree.fit(X_tr, y_tr)

print("depth:", tree.get_depth(), "  leaves:", tree.get_n_leaves())
print("training accuracy:", tree.score(X_tr, y_tr))
print("test accuracy:    ", round(tree.score(X_te, y_te), 3))

## Reading the learned tree

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)

small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)
print(export_text(small, feature_names=list(data.feature_names), show_weights=True))
print("training accuracy:", round(small.score(X_tr, y_tr), 3))
print("test accuracy:    ", round(small.score(X_te, y_te), 3))

## Why one tree overfits

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)
tree = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)

leaf_of = tree.apply(X_tr)              # the leaf each training patient ends in
sizes = np.bincount(leaf_of)            # patients per node id
sizes = sorted((int(s) for s in sizes if s > 0), reverse=True)
print("patients per leaf:", sizes)
print("leaves with 3 or fewer patients:", sum(s <= 3 for s in sizes))
print("leaves with exactly 1 patient:  ", sum(s == 1 for s in sizes))

## Limiting the size: choose `max_depth` by cross-validation

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)

print("depth  train   5-fold CV")
for d in range(1, 11):
    model = DecisionTreeClassifier(max_depth=d, random_state=0)
    cv = cross_val_score(model, X_tr, y_tr, cv=5)   # training rows only
    model.fit(X_tr, y_tr)
    print(f"{d:5d}  {model.score(X_tr, y_tr):.3f}   {cv.mean():.3f}")

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)

for d in [4, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    print(f"max_depth={d}: {m.get_n_leaves()} leaves, test accuracy {m.score(X_te, y_te):.3f}")

## Which features did the tree use?

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)

best = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
imp = best.feature_importances_                  # 30 numbers, they add up to 1
pairs = zip(data.feature_names, imp)
top = sorted(pairs, key=lambda t: -t[1])
for name, v in top[:5]:
    print(f"{name:25s} {v:.3f}")
print("features with importance 0:", sum(imp == 0), "of", len(imp))

### How the number is computed

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)
small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)

t = small.tree_                                  # the fitted tree, node by node
N = t.n_node_samples[0]                          # 455 patients at the root
total, contrib = 0.0, {}
for node in range(t.node_count):
    left, right = t.children_left[node], t.children_right[node]
    if left == -1:                               # a leaf asks no question
        continue
    n = t.n_node_samples[node]
    after = (t.n_node_samples[left] * t.impurity[left]
             + t.n_node_samples[right] * t.impurity[right]) / n
    gain = t.impurity[node] - after              # inside this group
    reach = n / N                                # share of all patients
    name = data.feature_names[t.feature[node]]
    print(f"{name:22s} gain {gain:.3f} x reach {reach:.3f} = {gain * reach:.3f}")
    contrib[name] = contrib.get(name, 0) + gain * reach
    total += gain * reach

for name, c in contrib.items():
    print(f"{name:22s} importance {c / total:.3f}")
print("scikit-learn's feature_importances_:")
for name, v in zip(data.feature_names, small.feature_importances_):
    if v > 0:
        print(f"{name:22s} importance {v:.3f}")

Back to the [Topic 17 lecture note](https://dongchul.kim/csci3320-fall2026/topic/18).